In [4]:
import json
import os

statsbomb_path = "../data/raw/statsbomb/data"

with open(os.path.join(statsbomb_path, "competitions.json")) as f:
    competitions = json.load(f)

# filter to only competition IDs that exist in the matches folder
available_ids = set(os.listdir(os.path.join(statsbomb_path, "matches")))

print(f"{'ID':<8} {'Competition':<35} {'Season'}")
print("-" * 70)
for c in competitions:
    if str(c['competition_id']) in available_ids:
        print(f"{c['competition_id']:<8} {c['competition_name']:<35} {c['season_name']}")

ID       Competition                         Season
----------------------------------------------------------------------
9        1. Bundesliga                       2023/2024
9        1. Bundesliga                       2015/2016
1267     African Cup of Nations              2023
16       Champions League                    2018/2019
16       Champions League                    2017/2018
16       Champions League                    2016/2017
16       Champions League                    2015/2016
16       Champions League                    2014/2015
16       Champions League                    2013/2014
16       Champions League                    2012/2013
16       Champions League                    2011/2012
16       Champions League                    2010/2011
16       Champions League                    2009/2010
16       Champions League                    2008/2009
16       Champions League                    2006/2007
16       Champions League                    2004/2005
16

Use results.csv goals for Elo computation across all 45k matches, but then train XGBoost on the StatsBomb subset using real xG as target. Elo features come from the full history, xG labels come from StatsBomb only.


ELO from results.csv xG from statsbomb

load results.csv
filter: date < 2022-11-20
replay all matches chronologically
maintain running elo dict {team: elo_rating}
at each match, snapshot {home_elo, away_elo} BEFORE updating
update elo after result
save full elo history to data/processed/elo_history.csv

load matches from competition 43 (WC) seasons: 1958,1962,1970,1974,1986,1990,2018
load matches from competition 55 (Euro) seasons: 2020
load matches from competition 223 (Copa) seasons: 2024
load matches from competition 1267 (AFCON) seasons: 2023
for each match, sum shot xG per team → home_xG, away_xG
output: {date, home_team, away_team, home_xG, away_xG}
save to data/processed/statsbomb_match_xg.csv

join statsbomb_match_xg with elo_history on date + teams
features: home_elo, away_elo, elo_diff, tournament_type
target: home_xG, away_xG
save to data/processed/training_set.csv

train two regressors: home_xG_model, away_xG_model
features: home_elo, away_elo, elo_diff, tournament_type
validate on WC 2022 matches
report MAE on predicted vs actual xG
save models to models/home_xg_model.json, away_xg_model.json

In [5]:
import pandas as pd
import numpy as np
import json
import os
import glob
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

# Paths
DATA_RAW       = "../data/raw"
DATA_PROCESSED = "../data/processed"
STATSBOMB_PATH = "../data/raw/statsbomb/data"

# Temporal cutoffs
TRAIN_CUTOFF = "2022-11-19"
VAL_START    = "2022-11-20"
VAL_END      = "2022-12-18"

# StatsBomb competitions to use
STATSBOMB_COMPS = {
    43:  ["2018", "1990", "1986", "1974", "1970", "1962", "1958"],  # FIFA World Cup (exclude 2022 - validation only)
    55:  ["2020"],                                                    # UEFA Euro
    223: ["2024"],                                                    # Copa America
    1267:["2023"],                                                    # AFCON
}

# WC 2022 StatsBomb - validation only
WC2022_COMP_ID = 43
WC2022_SEASON  = "2022"

In [24]:
results = pd.read_csv("../data/raw/kaggle/results.csv")
results['date'] = pd.to_datetime(results['date'])
train = results[results['date']<=TRAIN_CUTOFF]
val = results[
    (results['date'] >= VAL_START) & 
    (results['date'] <= VAL_END) & 
    (results['tournament'] == 'FIFA World Cup')
]


print(val.shape)
print(train.shape)

(64, 9)
(45700, 9)


In [ ]:
K = 32
DEFAULT_ELO = 1500

elo = {}  # {team_name: current_elo}
elo_records = []  # snapshots before each match

def get_elo(team):
    return elo.get(team, DEFAULT_ELO)

